# Teste isolado — ABR (Aeroportos do Brasil)

Fonte candidata: **ABR - Aeroportos do Brasil**, associação que representa
os aeroportos federais concedidos à iniciativa privada (59 terminais, 13
concessionárias). Setor Transporte (aviação/infraestrutura aeroportuária).
Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem de imprensa (título, data, link)
2. Extração do texto completo de uma notícia individual

## Confirmado antes de assumir

WordPress + Elementor confirmado, mas a listagem usa um plugin de
terceiros ("Themesflat Addons for Elementor", `tf-post.css`/`tf-post.js`)
— não é o loop nativo do tema nem os page builders já vistos em outras
fontes (GT3/jeg/tagDiv). Home (`/pt/`) não tem notícias — só estatísticas
institucionais e agenda de eventos; o menu tem duas seções candidatas,
**IMPRENSA** e **AIRPORTNEWS**. `https://abr.aero/pt/imprensa/` ("Assessoria
de Imprensa da ABR") é a listagem de releases oficiais da própria ABR —
usada aqui. `AIRPORTNEWS` devolveu HTTP 429 numa tentativa isolada de
conferência (rate limit pontual do servidor, não confirmado se é bloqueio
permanente) — não avaliada nesta Fase 1; se a ABR virar prioridade maior,
vale investigar depois como fonte adicional.

**Achado 1 — sem paginação**: `<nav class="navigation navigation-numeric-link">`
existe no HTML mas fica vazio — a página de imprensa mostra o histórico
completo (5 releases, de mar/2024 a dez/2025) numa página só. Sem
`max_paginas` necessário.

**Achado 2 — data em formato americano**: cada card tem a data em dois
lugares — `div.box-time` (dia + mês abreviado em português, ex. "01 dez")
e `li.post-date a`, este último como texto `"MM/DD/YYYY"` (formato dos
EUA, apesar do site inteiro estar em pt-BR). Confirmado comparando com a
URL do próprio post, que usa o padrão `/pt/AAAA/MM/DD/`
(`/pt/2025/12/01/` para "01 dez" → dezembro, dia 1) — por isso o parser
abaixo lê `mes, dia, ano` nessa ordem, não `dia, mes, ano` como na maioria
das outras fontes do projeto.

**Achado 3 — página individual tem um único `<h1>`**: diferente de
ABRACE/AESBE/AGERGS (que tinham `<h1>` duplicado por causa de headers de
template ou widgets laterais), aqui `extrair_titulo_h1()` genérico
funciona sem ajuste. Texto completo em
`.elementor-widget-theme-post-content` — mesmo seletor já usado por Trata
Brasil/AESBE, já presente em `SELETORES_CONTEUDO` do dispatcher genérico.

In [ ]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import re
import time
import random
import urllib.parse
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [ ]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://abr.aero/pt/imprensa/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_DATA_ABR = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")

In [ ]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    # Mesmos headers já usados no dispatcher genérico.
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a página de imprensa (sem paginação)

Cards em `div.tf-posts.grid > div.column`, título/link em `h2.title a`,
data em `li.post-date a` (texto `"MM/DD/YYYY"` — ver Achado 2).

In [ ]:
def listar_abr(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()

    for item in soup.select("div.tf-posts.grid > div.column"):
        tag_a = item.select_one("h2.title a")
        if not tag_a:
            continue
        url_item = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if url_item in vistos:
            continue
        vistos.add(url_item)

        data_publicacao = None
        tag_data = item.select_one("li.post-date a")
        if tag_data:
            m = PADRAO_DATA_ABR.search(tag_data.get_text(strip=True))
            if m:
                mes, dia, ano = m.groups()
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({
            "titulo": tag_a.get_text(strip=True),
            "url": url_item,
            "published_at": data_publicacao,
        })

    return itens

In [ ]:
html_listagem = baixar_pagina(SITE_URL)
itens = listar_abr(html_listagem, SITE_URL) if html_listagem else []

print(f"{len(itens)} releases listados.\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 100)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

`.elementor-widget-theme-post-content` — mesmo seletor Elementor já usado
por Trata Brasil/AESBE no dispatcher genérico. `extrair_titulo_h1()`
genérico funciona sem ajuste (página tem um único `<h1>`, ver Achado 3).

In [ ]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".elementor-widget-theme-post-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(" ", strip=True) if h1 else None


def extrair_noticia_abr(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    titulo = extrair_titulo_h1(html) or item["titulo"]

    return {
        "titulo": titulo,
        "url": item["url"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [ ]:
# Histórico pequeno (5 releases) -- abre todos, não precisa de amostra parcial.
detalhes = []
for item in itens:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_abr(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{len(itens)} releases abertos com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")

In [ ]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem sem paginação extrai título/data/link dos
5 releases, texto completo sai limpo via
`.elementor-widget-theme-post-content` (seletor Elementor já compartilhado
no dispatcher). Conteúdo é 100% institucional (releases de imprensa da
própria ABR — eventos, campanhas, posicionamento regulatório), sem
filtro de relevância aplicado, mesmo critério das demais fontes agregadoras
do projeto.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` — sem Selenium, sem parsing dependente de JS, sem
paginação para tratar. Precisa de uma `listar_abr()` própria (parser de
data em formato americano `MM/DD/YYYY`, ver Achado 2), mas o texto do
detalhe reaproveita `extrair_texto_generico()` com o seletor
`.elementor-widget-theme-post-content` já presente em `SELETORES_CONTEUDO`
— sem extrator próprio. `extrair_titulo_h1()` genérico também reaproveitado
sem ajuste.

Histórico pequeno (5 releases, página única, sem paginação) — diferente da
maioria das fontes recentes (ABAR/ABEGÁS/Trata Brasil etc.), aqui não há
necessidade de `max_paginas` nem de limitar a captura aos itens mais
recentes.